# 06 · Capstone — TravelMind, end to end
### memory + tools + multi-agent core, packaged on Runtime, observable

Everything from notebooks 00–05, assembled into one real agent.

**The scenario.** A traveler's flight is disrupted. TravelMind:
- checks the flight status (tool),
- already knows the traveler's seat/meal preferences (**Memory**),
- computes the exact EU261 compensation (**Code Interpreter** — no hallucinated math),
- finds an alternative flight and a nearby hotel (**specialist agents**),
- explains the policy plainly,
- and runs as a deployed, autoscaled service (**Runtime**) you can observe.

```mermaid
flowchart TB
    U[Traveler request] --> R[AgentCore Runtime<br/>app.py]
    R --> O[Orchestrator agent]
    O -->|memory hook| M[(AgentCore Memory<br/>traveler prefs)]
    O --> F[flight_agent<br/>status + alternatives]
    O --> P[policy_agent<br/>+ code interpreter math]
    O --> H[hotel_agent]
    R --> CW[CloudWatch traces/metrics]
```

We use the **agents-as-tools** orchestration here — it's the most controllable pattern, which is what you want in production. Region `us-east-1`; models Haiku 4.5 (specialists) and Sonnet 4.5 (orchestrator).


## Step 1 — (optional) create a Memory store once

Memory should be created **once**, out of band, and its id passed to the app via an environment variable — not recreated on every cold start. Run this once; keep the id.


In [ ]:
import os, json, boto3
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
REGION = "us-east-1"

from bedrock_agentcore.memory import MemoryClient
mc = MemoryClient(region_name=REGION)

memory = mc.create_memory_and_wait(
    name="TravelMindCapstoneMemory",
    strategies=[{"userPreferenceMemoryStrategy": {
        "name": "TravelerPrefs",
        "namespaces": ["travelmind/{actorId}/preferences"],
    }}],
    description="Traveler preferences for the TravelMind capstone",
    event_expiry_days=90,
)
MEMORY_ID = memory.get("memoryId") or memory.get("id")
print("MEMORY_ID =", MEMORY_ID)   # you'll pass this into the deployment as an env var

# Seed one preference so the agent has something to recall:
mc.create_event(memory_id=MEMORY_ID, actor_id="traveler-amelia", session_id="seed",
                messages=[("I always want an aisle seat and a vegetarian meal.", "USER"),
                          ("Noted: aisle seat, vegetarian meal.", "ASSISTANT")])

## Step 2 — build the brain locally and test it (before any deploy)

Compose the specialists, give the policy agent a Code Interpreter for exact math, wrap them as orchestrator tools, and attach a Memory hook scoped to the traveler. Test it right here — debugging locally is far faster than debugging a deployed container.


In [ ]:
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, AgentInitializedEvent
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_SONNET = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# --- tools (mock backends; swap for Gateway/real APIs in production) ---
@tool
def flight_status(flight_number: str) -> dict:
    """Current status of a flight by its number (e.g. BA117)."""
    return {"BA117": {"status": "Cancelled", "from": "LHR", "to": "JFK", "eu261_band_eur": 600}}\
        .get(flight_number.upper(), {"status": "Unknown"})

@tool
def find_alt_flight(origin: str, dest: str) -> list:
    """Find alternative flights between two airport codes."""
    return [{"flight": "BA179", "from": origin, "to": dest, "price_gbp": 420, "departs_in_h": 4}]

@tool
def find_hotel(city: str, nights: int) -> list:
    """Find hotels in a city for a number of nights."""
    return [{"name": "JFK Airport Inn", "city": city, "per_night_gbp": 150, "nights": nights}]

# --- memory hook (from notebook 03) ---
def _text_of(message) -> str:
    return "".join(b.get("text", "") for b in message.get("content", []) if isinstance(b, dict)).strip()

class MemoryHook(HookProvider):
    """Preload traveler preferences on init; persist each turn."""
    def __init__(self, client, memory_id, actor_id, session_id):
        self.client, self.memory_id = client, memory_id
        self.actor_id, self.session_id = actor_id, session_id
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AgentInitializedEvent, self.on_init)
        registry.add_callback(MessageAddedEvent, self.on_message)
    def on_init(self, event) -> None:
        recalled = self.client.retrieve_memories(
            memory_id=self.memory_id, namespace=f"travelmind/{self.actor_id}/preferences",
            query="seat and meal preferences", top_k=5)
        if recalled:
            event.agent.system_prompt += f"\n\nKnown traveler preferences: {recalled}"
    def on_message(self, event) -> None:
        role = event.message.get("role", "").upper(); text = _text_of(event.message)
        if role in ("USER", "ASSISTANT") and text:
            self.client.create_event(memory_id=self.memory_id, actor_id=self.actor_id,
                                     session_id=self.session_id, messages=[(text, role)])

# --- specialists (built once) ---
ci = AgentCoreCodeInterpreter(region=REGION)
flight_agent = Agent(model=MODEL_HAIKU, name="flight_agent", tools=[flight_status, find_alt_flight],
                     system_prompt="You handle flights only: status and alternatives. Be concise.")
policy_agent = Agent(model=MODEL_HAIKU, name="policy_agent", tools=[ci.code_interpreter],
                     system_prompt="Explain EU261 rules; when money is involved, RUN CODE for the exact figure.")
hotel_agent  = Agent(model=MODEL_HAIKU, name="hotel_agent", tools=[find_hotel],
                     system_prompt="You handle hotels only. Be concise.")

# --- specialists as orchestrator tools ---
@tool
def ask_flight_team(query: str) -> str:
    """Delegate a flight question to the flight specialist."""
    return str(flight_agent(query))
@tool
def ask_policy_team(query: str) -> str:
    """Delegate a compensation/policy question to the policy specialist."""
    return str(policy_agent(query))
@tool
def ask_hotel_team(query: str) -> str:
    """Delegate a hotel question to the hotel specialist."""
    return str(hotel_agent(query))

def build_orchestrator(actor_id: str, session_id: str) -> Agent:
    hooks = [MemoryHook(mc, MEMORY_ID, actor_id, session_id)] if MEMORY_ID else []
    return Agent(model=MODEL_SONNET, tools=[ask_flight_team, ask_policy_team, ask_hotel_team],
                 system_prompt=("You are TravelMind. For a disruption: confirm status, state exact "
                                "compensation, offer an alternative + hotel, honor known preferences. "
                                "Delegate to specialists, then give one clear answer."),
                 hooks=hooks)

# --- local test ---
orchestrator = build_orchestrator("traveler-amelia", "trip-tokyo-001")
print(str(orchestrator(
    "My flight BA117 LHR->JFK was cancelled. What am I owed, what's my alternative, "
    "and find me a hotel near JFK for 1 night. Remember my usual seat/meal."
)))

That single answer pulled four threads together: status, exact (code-computed) compensation, an alternative + hotel from specialists, and the traveler's recalled preferences. Now make it a service.


## Step 3 — package as `app.py` for Runtime

Same logic, wrapped in `BedrockAgentCoreApp`. The entrypoint reads `prompt`, `actor_id`, and `session_id` from the payload, builds an orchestrator scoped to that traveler, and returns the answer. `MEMORY_ID` comes from the environment (threaded in at deploy time).


In [ ]:
%%writefile app.py
"""TravelMind capstone — multi-agent disruption assistant for AgentCore Runtime."""
import os
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, AgentInitializedEvent
from strands_tools.code_interpreter import AgentCoreCodeInterpreter
from bedrock_agentcore.memory import MemoryClient

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
MEMORY_ID = os.environ.get("MEMORY_ID")            # threaded in at deploy time (env var)
MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
MODEL_SONNET = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

mc = MemoryClient(region_name=REGION) if MEMORY_ID else None

@tool
def flight_status(flight_number: str) -> dict:
    """Current status of a flight by its number (e.g. BA117)."""
    return {"BA117": {"status": "Cancelled", "from": "LHR", "to": "JFK", "eu261_band_eur": 600}}\
        .get(flight_number.upper(), {"status": "Unknown"})

@tool
def find_alt_flight(origin: str, dest: str) -> list:
    """Find alternative flights between two airport codes."""
    return [{"flight": "BA179", "from": origin, "to": dest, "price_gbp": 420, "departs_in_h": 4}]

@tool
def find_hotel(city: str, nights: int) -> list:
    """Find hotels in a city for a number of nights."""
    return [{"name": "JFK Airport Inn", "city": city, "per_night_gbp": 150, "nights": nights}]

def _text_of(message):
    return "".join(b.get("text", "") for b in message.get("content", []) if isinstance(b, dict)).strip()

class MemoryHook(HookProvider):
    def __init__(self, client, memory_id, actor_id, session_id):
        self.client, self.memory_id = client, memory_id
        self.actor_id, self.session_id = actor_id, session_id
    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AgentInitializedEvent, self.on_init)
        registry.add_callback(MessageAddedEvent, self.on_message)
    def on_init(self, event):
        recalled = self.client.retrieve_memories(
            memory_id=self.memory_id, namespace=f"travelmind/{self.actor_id}/preferences",
            query="seat and meal preferences", top_k=5)
        if recalled:
            event.agent.system_prompt += f"\n\nKnown traveler preferences: {recalled}"
    def on_message(self, event):
        role = event.message.get("role", "").upper(); text = _text_of(event.message)
        if role in ("USER", "ASSISTANT") and text:
            self.client.create_event(memory_id=self.memory_id, actor_id=self.actor_id,
                                     session_id=self.session_id, messages=[(text, role)])

ci = AgentCoreCodeInterpreter(region=REGION)
flight_agent = Agent(model=MODEL_HAIKU, name="flight_agent", tools=[flight_status, find_alt_flight],
                     system_prompt="You handle flights only: status and alternatives. Be concise.")
policy_agent = Agent(model=MODEL_HAIKU, name="policy_agent", tools=[ci.code_interpreter],
                     system_prompt="Explain EU261 rules; when money is involved, RUN CODE for the exact figure.")
hotel_agent  = Agent(model=MODEL_HAIKU, name="hotel_agent", tools=[find_hotel],
                     system_prompt="You handle hotels only. Be concise.")

@tool
def ask_flight_team(query: str) -> str:
    """Delegate a flight question to the flight specialist."""
    return str(flight_agent(query))
@tool
def ask_policy_team(query: str) -> str:
    """Delegate a compensation/policy question to the policy specialist."""
    return str(policy_agent(query))
@tool
def ask_hotel_team(query: str) -> str:
    """Delegate a hotel question to the hotel specialist."""
    return str(hotel_agent(query))

app = BedrockAgentCoreApp()

@app.entrypoint
def invoke(payload):
    actor = payload.get("actor_id", "anonymous")
    session = payload.get("session_id", "session-" + actor)
    hooks = [MemoryHook(mc, MEMORY_ID, actor, session)] if mc else []
    orchestrator = Agent(
        model=MODEL_SONNET,
        tools=[ask_flight_team, ask_policy_team, ask_hotel_team],
        system_prompt=("You are TravelMind. For a disruption: confirm status, state exact "
                       "compensation, offer an alternative + hotel, honor known preferences. "
                       "Delegate to specialists, then give one clear answer."),
        hooks=hooks,
    )
    return {"result": str(orchestrator(payload.get("prompt", "")))}

if __name__ == "__main__":
    app.run()

## Step 4 — test the container locally

```bash
python app.py     # serves http://localhost:8080

curl http://localhost:8080/ping
curl -X POST http://localhost:8080/invocations \
     -H "Content-Type: application/json" \
     -d '{"prompt": "BA117 LHR->JFK cancelled. Compensation, alternative, hotel near JFK 1 night?", "actor_id": "traveler-amelia", "session_id": "trip-tokyo-001"}'
```


## Step 5 — deploy to Runtime

`configure` records the package; `launch` builds the ARM64 container, pushes to ECR, provisions the role/endpoint, and threads `MEMORY_ID` in as an env var.

> **Role note:** the auto-created execution role needs Bedrock model access **and** Code Interpreter permissions (the policy agent uses the sandbox). If you drop the code-interpreter tool, you don't need that permission.


In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

runtime = Runtime()
runtime.configure(
    entrypoint="app.py",
    agent_name="travelmind_capstone",
    requirements=["strands-agents", "strands-agents-tools", "bedrock-agentcore"],
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region="us-east-1",
)
launch_result = runtime.launch(env_vars={"MEMORY_ID": MEMORY_ID})   # pass the memory id to the container
print(launch_result)

In [ ]:
resp = runtime.invoke({
    "prompt": "My flight BA117 LHR->JFK was cancelled. Compensation, an alternative, and a hotel near JFK for 1 night?",
    "actor_id": "traveler-amelia",
    "session_id": "trip-tokyo-001",
})
print(resp)

## Step 6 — observe it

AgentCore emits traces and metrics to **CloudWatch (GenAI Observability)** automatically — per-invocation latency, token usage, tool calls, errors. Open the CloudWatch GenAI Observability view for your runtime to see sessions and spans without adding instrumentation.

To go further on quality:

```bash
agentcore add evaluator      # LLM-as-a-judge scoring of responses
agentcore add online-eval    # continuous evaluation on live traffic
agentcore deploy
```

## Step 7 — clean up

```python
runtime.destroy()
# and delete the memory store when done:
# mc.delete_memory(memory_id=MEMORY_ID)
```


## What you built — and the map back

| Capability | Notebook | In the capstone |
|---|---|---|
| Three ways to build an agent | 01 | Strands specialists + orchestrator |
| Runtime hosting | 02 | `app.py` deployed, sessions, env vars |
| Memory | 03 | `MemoryHook`, traveler preferences |
| Tools (Code Interpreter) | 04 | exact EU261 compensation math |
| Tools (Gateway/Browser/Identity) | 04 | the swap-in path for real APIs + auth |
| Multi-agent orchestration | 05 | agents-as-tools coordination |

**Production hardening checklist (next steps, not in scope here):**
- Swap mock tools for **Gateway**-fronted real APIs; put secrets in **Identity**, never in code.
- Add a **JWT authorizer** in front of the runtime (inbound identity).
- Move to the **`@aws/agentcore` CLI** project workflow for CI/CD.
- Turn on **evaluators / online-eval**; watch CloudWatch dashboards.
- Use **`S3SessionManager`** (not file-based) if you persist sessions across concurrent instances.

**Skeptic's last word.** This capstone is deliberately one orchestrator + three specialists with mock tools. Don't ship the mocks. The architecture is real; the data isn't. Replace tools with governed APIs, add auth, and measure quality before you call it production.

You've gone scratch → foundations → intermediate → a deployed multi-agent system. That's AgentCore end to end.
